# YOLO App : เลือกรูปแล้วทาย

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import tkinter as tk
from tkinter import Button, Label, Scale, filedialog, messagebox
from PIL import Image, ImageTk
from ultralytics import YOLO

In [ ]:
class PredictApp:
    def __init__(self, root, models):
        self.root = root
        self.root.title("YOLO Predict")
        self.models = models
        self.file_path = None

        self.image_label = Label(self.root, text="ยังไม่ได้เลือกรูป", width=80, height=25, bg="gray")
        self.image_label.pack(padx=10, pady=10)

        Button(self.root, text="Open Image", command=self.open_image).pack()
        for name in self.models:
            Button(self.root, text=f"Predict {name}", command=lambda n=name: self.predict(n)).pack()

        self.conf_scale = Scale(self.root, from_=0.05, to=0.95, resolution=0.05,
                                orient="horizontal", label="Confidence", length=250)
        self.conf_scale.set(0.35)
        self.conf_scale.pack()

        self.result_label = Label(self.root, text="", font=("Tahoma", 12))
        self.result_label.pack(pady=10)

    def show(self, img):
        img.thumbnail((640, 480))
        self.photo = ImageTk.PhotoImage(img)
        self.image_label.config(image=self.photo, width=img.width, height=img.height)

    def open_image(self):
        self.file_path = filedialog.askopenfilename(
            filetypes=[("Image files", "*.jpg *.jpeg *.png"), ("All files", "*.*")]
        )
        if self.file_path:
            self.show(Image.open(self.file_path))
            self.result_label.config(text="")

    def predict(self, name):
        if not self.file_path:
            messagebox.showerror("Error", "กรุณาเลือกรูปก่อน")
            return
        try:
            r = self.models[name].predict(self.file_path, conf=self.conf_scale.get())[0]
            self.show(Image.fromarray(r.plot()[:, :, ::-1].copy()))

            text = [f"{r.names[int(b.cls)]}  {float(b.conf):.2f}" for b in r.boxes]
            self.result_label.config(text="\n".join(text) if text else "ไม่พบวัตถุ")
        except Exception as e:
            messagebox.showerror("Error", f"Error predicting image: {e}")

In [ ]:
models = {
    "Rectangle": YOLO("../Weights/rectangle_best.pt"),
    "Polygon": YOLO("../Weights/polygon_best.pt"),
}

root = tk.Tk()
app = PredictApp(root, models)
root.mainloop()